In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.model_selection import train_test_split


In [3]:
t = pd.read_csv('train.csv')

In [4]:
t['comment'] = t['comment'].fillna('')

In [5]:
train_df, test_df = train_test_split(t, random_state=2306, stratify=t['label'], test_size=.4)

In [6]:
vc_a = pd.DataFrame([train_df['label'].value_counts(),test_df['label'].value_counts()], index=['train', 'test'])

In [7]:
train_df, test_df = train_test_split(t, random_state=2306, test_size=.4)

In [8]:
vc_b = pd.DataFrame([train_df['label'].value_counts(),test_df['label'].value_counts()], index=['train', 'test'])

In [9]:
np.abs(vc_a.loc['test']/vc_a.loc['test'].sum()-vc_b.loc['test']/vc_b.loc['test'].sum())

label
0    0.000164
2    0.000253
1    0.000606
3    0.000189
Name: test, dtype: float64

In [10]:
train_df.columns

Index(['created_date', 'post_id', 'emoticon_1', 'emoticon_2', 'emoticon_3',
       'upvote', 'downvote', 'if_1', 'if_2', 'race', 'religion', 'gender',
       'disability', 'comment', 'label'],
      dtype='object')

In [13]:
train_df, test_df = train_test_split(t, random_state=2306, stratify=t['label'], test_size=.4)

In [14]:
x_train, y_train = train_df[['created_date', 'post_id', 'emoticon_1', 'emoticon_2', 'emoticon_3',
       'upvote', 'downvote', 'if_1', 'if_2', 'race', 'religion', 'gender',
       'disability', 'comment']], train_df['label']

In [15]:
x_test, y_test = test_df[['created_date', 'post_id', 'emoticon_1', 'emoticon_2', 'emoticon_3',
       'upvote', 'downvote', 'if_1', 'if_2', 'race', 'religion', 'gender',
       'disability', 'comment']], test_df['label']

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

In [17]:
x_train = x_train.drop(columns=["created_date"])
x_test = x_test.drop(columns=["created_date"])


In [18]:
text_x_train = x_train["comment"]
text_x_test = x_test["comment"]

In [19]:

text_x_train = text_x_train.fillna("")
text_x_test = text_x_test.fillna("")

x_train = x_train.drop(columns=["comment"])
x_test = x_test.drop(columns=["comment"])

In [20]:
categorical_cols = ["race", "religion", "gender", "disability"]

numerical_cols = [
    "post_id", "emoticon_1", "emoticon_2", "emoticon_3",
    "upvote", "downvote", "if_1", "if_2"
]

categorical_pipeline = [
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
]

numerical_pipeline = [
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
]

from sklearn.pipeline import Pipeline

categorical_pipeline = Pipeline(categorical_pipeline)
numerical_pipeline = Pipeline(numerical_pipeline)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_pipeline, categorical_cols),
        ("num", numerical_pipeline, numerical_cols)
    ],
    remainder="passthrough"
)


In [21]:
x_train_tabular = preprocessor.fit_transform(x_train)


In [22]:
x_test_tabular = preprocessor.transform(x_test)


In [23]:
import re
def normalize_text(text):
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\s+', '  ', text).strip()
    return text

text_x_train_norm = text_x_train.apply(normalize_text)
text_x_test_norm = text_x_test.apply(normalize_text)


In [24]:
vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)

tf_idf_train = vectorizer.fit_transform(text_x_train_norm)
tf_idf_test = vectorizer.transform(text_x_test_norm)


In [25]:
X_train_final = hstack([x_train_tabular, tf_idf_train])
X_test_final = hstack([x_test_tabular, tf_idf_test])

In [26]:
X_train_final.sum()

np.float64(904262.9332531855)

In [27]:
### SVD Section

In [28]:
from sklearn.decomposition import TruncatedSVD

In [29]:
svd = TruncatedSVD(n_components=300, random_state=2306)

X_train_reduced = svd.fit_transform(X_train_final)
X_test_reduced = svd.transform(X_test_final)

In [30]:
### Grid Search - RF

In [31]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

In [43]:
rf = RandomForestClassifier(random_state=2306)
param_dist = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, 15]
}


randomized_search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=5,
    cv=3,
    random_state=2306,
    n_jobs=-1
)

randomized_search.fit(X_train_reduced, y_train)
best_params = randomized_search.best_params_


In [44]:
best_params["n_estimators"]

200

In [ ]:
### ADABoost

In [33]:
from sklearn.ensemble import AdaBoostClassifier

model = AdaBoostClassifier(
    n_estimators=50,
    random_state=2306
)
model.fit(X_train_reduced, y_train)

errors = model.estimator_errors_
variance = np.var(errors)
variance

np.float64(0.005810087377281584)

### RF

In [34]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=2306
)

In [35]:
model.fit(X_train_reduced, y_train)


RandomForestClassifier(max_depth=10, random_state=2306)

In [36]:
importances = model.feature_importances_


In [37]:
np.argmax(importances)

np.int64(4)

In [ ]:
### MLP

In [38]:
# number of input features
N = X_train_reduced.shape[1]

# compute weights (excluding biases)
total_weights = (
    N * 128 +
    128 * 64 +
    64 * 32 +
    32 * 4
)

print(total_weights)

48768


In [39]:
from sklearn.neural_network import MLPClassifier

# Initialize the model
model = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    max_iter=5,
    batch_size=32,
    random_state=2306
)

# Train the model
model.fit(X_train_reduced, y_train)

# Get final loss
final_loss = model.loss_

print(round(final_loss, 4))

0.3271


c:\Users\rramk\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(


### MLP2

In [40]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score
import numpy as np

# Model A (default alpha)
model_a = MLPClassifier(
    hidden_layer_sizes=(100,),
    max_iter=5,
    random_state=2306
)

# Model B (strong regularization)
model_b = MLPClassifier(
    hidden_layer_sizes=(100,),
    alpha=1.0,
    max_iter=5,
    random_state=2306
)

# Train models
model_a.fit(X_train_reduced, y_train)
model_b.fit(X_train_reduced, y_train)

# Predictions on training set
pred_a = model_a.predict(X_train_reduced)
pred_b = model_b.predict(X_train_reduced)

# Macro F1 scores
f1_a = f1_score(y_train, pred_a, average="macro")
f1_b = f1_score(y_train, pred_b, average="macro")

# Absolute difference
diff = abs(f1_a - f1_b)

print(round(diff, 4))

c:\Users\rramk\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rramk\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(


0.1327


In [42]:
from sklearn.metrics import confusion_matrix

# Assuming the Q7 MLP model is stored in variable `model`
# And validation data is: X_val_reduced, y_val

# Predict on validation set
y_pred = model.predict(X_test_reduced)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Sum of off-diagonal elements = total misclassified
misclassified = cm.sum() - np.trace(cm)

# Proportion of misclassified
prop_misclassified = misclassified / y_test.shape[0]

print(round(prop_misclassified, 4))

0.1126
